In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Processed_data/ML_ready_dataset.csv")
df.head(3)

,price,freight_value,order_status,review_score,delivery_delay_days,has_review,risk_type,purchase_approved_gap,approved_shipped_gap,courier_risk_flag,...,product_category_name_english_housewares,product_category_name_english_luggage_accessories,product_category_name_english_office_furniture,product_category_name_english_perfumery,product_category_name_english_pet_shop,product_category_name_english_sports_leisure,product_category_name_english_stationery,product_category_name_english_telephony,product_category_name_english_toys,product_category_name_english_watches_gifts
0,58.9,13.29,1,5.0,-9.0,1,1,0.0,6.0,0,...,False,False,False,False,False,False,False,False,False,False
1,239.9,19.93,1,4.0,-3.0,1,1,0.0,8.0,0,...,False,False,False,False,True,False,False,False,False,False
2,199.0,17.87,1,5.0,-14.0,1,1,0.0,1.0,0,...,False,False,False,False,False,False,False,False,False,False


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103771 entries, 0 to 103770
Data columns (total 47 columns):
 #   Column                                                  Non-Null Count   Dtype  
---  ------                                                  --------------   -----  
 0   price                                                   103771 non-null  float64
 1   freight_value                                           103771 non-null  float64
 2   order_status                                            103771 non-null  int64  
 3   review_score                                            103771 non-null  float64
 4   delivery_delay_days                                     103771 non-null  float64
 5   has_review                                              103771 non-null  int64  
 6   risk_type                                               103771 non-null  int64  
 7   purchase_approved_gap                                   103771 non-null  float64
 8   approved_shipped_gap    

In [4]:
print(list(df.columns))

['price', 'freight_value', 'order_status', 'review_score', 'delivery_delay_days', 'has_review', 'risk_type', 'purchase_approved_gap', 'approved_shipped_gap', 'courier_risk_flag', 'payment_installments', 'payment_value', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_state', 'customer_zip_code_prefix', 'customer_state', 'delivery_days', 'delivery_anomaly', 'invalid_timeline', 'delay_days_category', 'payment_type_credit_card', 'payment_type_debit_card', 'payment_type_voucher', 'product_category_name_english_auto', 'product_category_name_english_baby', 'product_category_name_english_bed_bath_table', 'product_category_name_english_computers_accessories', 'product_category_name_english_cool_stuff', 'product_category_name_english_electronics', 'product_category_name_english_fashion_bags_accessories', 'product_category_name_english_furniture_decor', 'product_category_name_english_garden_tools', 'product_category_name_english

In [5]:
drop_cols = [
    "delivery_delay_days",
    "delivery_anomaly",
    "invalid_timeline",
    "review_score",
    "has_review",
    "order_status",
    "seller_zip_code_prefix",
    "customer_zip_code_prefix"
]
df = df.drop(columns=drop_cols)

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [7]:
x = df.drop(['delay_days_category'],axis='columns')
y = df[["delay_days_category"]]

In [8]:
y.head(3)

,delay_days_category
0,1
1,1
2,1


In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

C:\Users\ASMIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


### *TRAIN TEST SPLIT* 

In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [11]:
def find_best_model_using_gridsearchcv(X, y):
    
    models = {
        "logistic_regression": {
            "pipeline": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000, multi_class="auto"))
            ]),
            "params": {
                "model__C": [0.1, 1, 5],
                "model__solver": ["lbfgs"]
            }
        },

        "random_forest": {
            "pipeline": Pipeline([
                ("model", RandomForestClassifier(random_state=42))
            ]),
            "params": {
                "model__n_estimators": [200, 300],
                "model__max_depth": [None, 10, 20]
            }
        },

        "xgboost": {
            "pipeline": Pipeline([
                ("model", XGBClassifier(
                    objective="multi:softprob",
                    eval_metric="mlogloss",
                    random_state=42
                ))
            ]),
            "params": {
                "model__n_estimators": [200, 300],
                "model__max_depth": [4, 6],
                "model__learning_rate": [0.05, 0.1]
            }
        }
    }

    scores = []
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for model_name, config in models.items():
        gs = GridSearchCV(
            config["pipeline"],
            config["params"],
            cv=cv,
            scoring="neg_log_loss",   # IMPORTANT for probabilistic models
            n_jobs=-1
        )

        gs.fit(X, y)

        scores.append({
            "model": model_name,
            "best_score": gs.best_score_,
            "best_params": gs.best_params_
        })

    return pd.DataFrame(scores)

results = find_best_model_using_gridsearchcv(X_train, y_train)
print(results)

C:\Users\ASMIT\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1264: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


                 model  best_score  \
0  logistic_regression   -0.147440   
1        random_forest   -0.136059   
2              xgboost   -0.126555   

                                         best_params  
0          {'model__C': 1, 'model__solver': 'lbfgs'}  
1  {'model__max_depth': 20, 'model__n_estimators'...  
2  {'model__learning_rate': 0.1, 'model__max_dept...  


### *All three models show comparable performance based on log loss, with 'Random Forest' performing marginally better.*

## *XG Boost*

In [13]:
pipe = Pipeline([
    ("model", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42
    ))
])

In [14]:
param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1]
}

In [15]:
gs = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="neg_log_loss",
    n_jobs=-1
)

gs.fit(X_train, y_train)

,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'model__learning_rate': [0.05, 0.1], 'model__max_depth': [4, 6], 'model__n_estimators': [200, 300]}"
,scoring,'neg_log_loss'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'multi:softprob'


In [17]:
from sklearn.metrics import log_loss

best_model_XG= gs.best_estimator_
y_proba_XG = best_model_XG.predict_proba(X_test)

print("Test Log Loss:", log_loss(y_test, y_proba_XG))

Test Log Loss: 0.12643642944956726


### *The model achieved a test log loss of '0.126', indicating highly reliable and well-calibrated probability predictions for delivery delay risk.*

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import log_loss

## *RANDOM FOREST*

In [26]:
# Pipeline
pipe = Pipeline([
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ))
])

In [27]:
# Hyperparameter grid
param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

In [28]:
# GridSearch
gs = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring="neg_log_loss",   # probabilistic metric
    n_jobs=-1
)
# Train
gs.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [10, 20, ...], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], 'model__n_estimators': [200, 300]}"
,scoring,'neg_log_loss'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,300


In [29]:
from sklearn.metrics import log_loss

best_model = gs.best_estimator_
y_proba = best_model.predict_proba(X_test)

print("Test Log Loss:", log_loss(y_test, y_proba))

Test Log Loss: 0.13693397647530742


### *The 'Randomforest Classifier' model achievs 0.13 log loss, which indicates a good predictions but the score is slightly low in compare with the score of 'XGBoost Classifier. So final selected model is "XGBoost Classifier"*

### *LOAD THE MODEL INTO PICKLE FILE*

In [18]:
import pickle

with open('delivery_delay_prediction_model.pkl', 'wb') as f:
    pickle.dump(best_model_XG, f)